# LLM Inference Benchmark (SGLang)

This notebook benchmarks SGLang inference across three configurations, each run
in its own cell with a **fresh SGLang server instance**:

| Test | Configuration | Description |
|------|--------------|-------------|
| 1 | **Baseline** | Standard SGLang with RadixAttention (automatic prefix caching) |
| 2 | **No RadixAttention** | RadixAttention disabled to measure its benefit |
| 3 | **Torch Compile** | torch.compile optimization for compute-bound speedups |

## Metrics collected per test
- **Throughput**: Output tokens/s, total tokens/s, requests/s
- **Latency**: TTFT p50/p95/p99, ITL p50/p95/p99, request latency
- **Goodput**: Successful tokens/s, success rate, SLA compliance
- **KV Cache**: Utilization %, prefix cache hit rate

## Workloads
- `random_short` — Short prompts (<100 tokens), tests raw decode speed
- `random_medium` — Medium prompts (100-500 tokens), balanced workload
- `long_context` — Long prompts (500-1500 tokens), stresses prefill throughput
- `mixed_context` — 50% very-long (≥800 tokens) + 50% very-short (<50 tokens); shows chunked-prefill head-of-line blocking
- `shared_prefix` — Same prefix + different questions, tests KV cache reuse
- `repetitive` — Repetitive patterns, tests n-gram speculation
- `code` — Code generation, tests structured output speculation

Each test saves its own DataFrame and CSV. After all tests, comprehensive
comparison graphs are generated.

## SGLang Key Features Tested
- **RadixAttention**: Automatic KV cache reuse across requests sharing prefixes
- **torch.compile**: JIT compilation for reduced kernel launch overhead
- **FlashInfer backend**: High-performance attention kernels for A100

---
## 1. Install Dependencies

In [ ]:
# Install SGLang with all dependencies (includes FlashInfer for A100)
!pip install "sglang[all]>=0.4.0" -q
!pip install "openai>=1.51.0" "transformers>=4.44.2" \
    "tokenizers>=0.19.1" "accelerate" "GPUtil" "httpx>=0.27" "pandas" \
    "psutil" "torch" "gpustat" "nvidia-ml-py" "seaborn" "matplotlib" \
    "huggingface_hub" "datasets" "aiohttp" "nest_asyncio" -q

# Verify SGLang install
import sglang
print(f"SGLang version: {sglang.__version__}")

## 2. Hugging Face Authentication

In [ ]:
import os

os.environ["HF_TOKEN"] = 'YOUR_HF_TOKEN_HERE'

from huggingface_hub import login
login(token=os.environ["HF_TOKEN"])

## 3. Download Models

Downloads the target model. The draft model is also downloaded for parity
with the vLLM benchmark setup (not used by SGLang tests).

In [ ]:
from huggingface_hub import snapshot_download

target_model = "meta-llama/Llama-3.2-3B-Instruct"

print(f"Downloading {target_model}...")
snapshot_download(target_model)
print("Model downloaded.")

## 4. Upload Benchmark Module

Upload the `llm_benchmark/` folder to your Colab environment.

In [ ]:
import os

required_files = [
    "llm_benchmark/__init__.py",
    "llm_benchmark/benchmark_core.py",
    "llm_benchmark/server.py",
    "llm_benchmark/workloads.py",
    "llm_benchmark/metrics.py",
    "llm_benchmark/stats.py",
    "llm_benchmark/runner.py",
]

missing = [f for f in required_files if not os.path.exists(f)]
if missing:
    print("Missing files:")
    for f in missing:
        print(f"  - {f}")
    print("\nPlease upload the llm_benchmark folder.")
else:
    print("llm_benchmark module found. All files present.")

## 4b. Dataset File Configuration

Both vLLM and SGLang notebooks share the same prompts via a JSON dataset file.
- If the file exists, prompts are loaded from it (ensuring identical inputs across engines).
- If it does not exist, prompts are generated from the Dolly dataset and saved to the file.

Set `DATASET_FILE` below to control the path. Upload the file to Colab, or
leave the default to auto-generate on first run.

In [ ]:
import os

# Path to the shared dataset file.
# Both vLLM and SGLang notebooks should point to the SAME file.
DATASET_FILE = "benchmark_prompts.json"

# Set to False to disable KV cache metric polling during benchmarks.
# When disabled, KV cache columns will report 0 / NaN instead of
# polling the server's /metrics endpoint every second.
COLLECT_KV_STATS = False

if os.path.exists(DATASET_FILE):
    print(f"Dataset file found: {DATASET_FILE}")
    print(f"  Size: {os.path.getsize(DATASET_FILE) / 1024:.1f} KB")
    print("  Prompts will be loaded from this file.")
else:
    print(f"Dataset file not found: {DATASET_FILE}")
    print("  Prompts will be generated and saved to this file.")
    print("  Tip: Upload a previously generated file to use identical prompts across runs.")

## 5. Initialize Benchmark Environment

This cell:
1. Detects GPU and selects optimal dtype
2. Loads the tokenizer (shared across all tests)
3. Loads workloads from a shared dataset file (or generates & saves one)
4. Defines the `run_single_config()` helper used by each test cell

In [ ]:
import time
import pandas as pd
import numpy as np
from transformers import AutoTokenizer

from llm_benchmark import (
    detect_gpu,
    pick_dtype,
    ServerConfig,
    ServerProcess,
    WorkloadGenerator,
    WORKLOAD_CONFIGS,
    run_workload,
    TestConfig,
    SGLANG_TEST_CONFIGS,
    DEFAULT_BATCH_SIZES,
    StatisticsReporter,
)

# ---- Shared state (computed once, used by every test cell) ----
MODEL_ID = "meta-llama/Llama-3.2-3B-Instruct"
BATCH_SIZES = DEFAULT_BATCH_SIZES          # [1, 8, 32, 64]
WORKLOAD_NAMES = list(WORKLOAD_CONFIGS.keys())

# Per-workload batch-size caps (prevents OOM / server stalls).
# SGLang uses a fixed-size static KV cache pool (--mem-fraction-static).
# Unlike vLLM's dynamic allocation, SGLang crashes instead of rejecting
# requests when the pool is exhausted.
MAX_BATCH = {
    "shared_prefix": 32,
    "repetitive": 8,
    "long_context": 32,
    "mixed_context": 32,   # half long (>=1500 tok) prompts stress static KV pool
}

gpu_info = detect_gpu()
dtype = pick_dtype(gpu_info)
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, use_fast=True)

workload_gen = WorkloadGenerator(MODEL_ID)

# --- Load or generate workloads ---
if os.path.exists(DATASET_FILE):
    workload_gen.load_workloads(DATASET_FILE)
    print(f"\n✅ Loaded prompts from {DATASET_FILE}")
else:
    workload_gen.prepare_workloads(
        n_random=max(BATCH_SIZES),
        n_shared=max(BATCH_SIZES),
    )
    workload_gen.save_workloads(DATASET_FILE)
    print(f"\n💾 Generated and saved prompts to {DATASET_FILE}")

print(f"\nGPU: {gpu_info['name']} ({gpu_info['memory_gb']:.1f} GB, CC {gpu_info['cc']})")
print(f"Dtype: {dtype}")
print(f"Batch sizes: {BATCH_SIZES}")
print(f"Workloads: {list(workload_gen._workloads.keys())}")


def run_single_config(
    config: TestConfig,
    batch_sizes=None,
    workload_names=None,
    warmup: int = 3,
) -> pd.DataFrame:
    """
    For *config*, start a FRESH SGLang server for EACH workload, run every
    batch size against it, then shut it down before moving to the next
    workload.  This isolates workloads so a crash, OOM, or KV pool
    fragmentation in one cannot affect the next.

    All rows still go into a single reporter and one CSV — same output
    shape as before.  Cost: one extra server start per workload (~30-60 s
    each on an L4), so 9 workloads ≈ 9 startups per config.
    """
    batch_sizes = batch_sizes or BATCH_SIZES
    workload_names = workload_names or WORKLOAD_NAMES
    reporter = StatisticsReporter()

    # -- build server launch args (same for every workload) --
    extra_args = list(config.extra_args)
    # Note: spec_config is vLLM-specific; SGLang configs use extra_args instead
    if config.spec_config:
        extra_args.extend(["--speculative-config", config.spec_config])

    for wl_name in workload_names:
        wl_cfg = WORKLOAD_CONFIGS[wl_name]
        prompts = workload_gen.get_workload(wl_name)
        if not prompts:
            print(f"\n=== {wl_name} — SKIPPED (no prompts loaded) ===")
            continue
        cap = MAX_BATCH.get(wl_name, float("inf"))

        print(f"\n{'=' * 60}")
        print(f"Workload: {wl_name} — {wl_cfg.description}")
        print(f"{'=' * 60}")

        # Fresh server for this workload
        server_cfg = ServerConfig(
            engine=config.engine,
            model_id=MODEL_ID,
            port=8000,
            dtype=dtype,
            gpu_mem_util=config.gpu_mem_util,
            max_model_len=8192,
            extra_args=extra_args,
        )
        server = ServerProcess(server_cfg)
        try:
            server.start()
        except Exception as e:
            print(f"  Server failed to start for {wl_name}: {e}")
            continue

        try:
            # Warmup the fresh server
            if warmup > 0:
                print("  Warming up...")
                wp = workload_gen.get_workload("random_short")[:warmup]
                _ = run_workload(
                    base_url=server_cfg.base_url(),
                    model=MODEL_ID,
                    prompts=wp,
                    tokenizer=tokenizer,
                    concurrency=warmup,
                    max_tokens=32,
                    collect_kv_stats=COLLECT_KV_STATS,
                )

            # Run every batch size on this server
            for bs in batch_sizes:
                if bs > cap:
                    print(f"    Batch {bs:>4d} ... skipped (cap {int(cap)} for {wl_name})")
                    continue

                print(f"    Batch {bs:>4d} ... ", end="", flush=True)
                try:
                    bm = run_workload(
                        base_url=server_cfg.base_url(),
                        model=MODEL_ID,
                        prompts=prompts[:bs],
                        tokenizer=tokenizer,
                        concurrency=bs,
                        max_tokens=wl_cfg.max_tokens,
                        temperature=wl_cfg.temperature,
                        collect_kv_stats=COLLECT_KV_STATS,
                    )
                    reporter.add_run(
                        config_name=config.name,
                        workload_name=wl_name,
                        batch_size=bs,
                        batch_metrics=bm,
                    )
                    tp = bm.throughput_output_tok_s
                    print(f"{tp:,.1f} tok/s  ({bm.successful_requests}/{bm.total_requests} ok)")
                except Exception as e:
                    print(f"ERROR: {e}")
        finally:
            server.stop()

    df = reporter.to_dataframe()
    csv_name = f"results_{config.name}.csv"
    df.to_csv(csv_name, index=False)
    print(f"\n  Saved {len(df)} rows → {csv_name}")
    return df


---
## 6. Test 1 — Baseline (RadixAttention ON)

Starts a fresh SGLang server with default settings. RadixAttention is
enabled by default, providing automatic prefix caching. This is SGLang's
flagship optimization — requests sharing common prefixes reuse KV cache
entries via a radix tree.

In [ ]:
# --- Quick isolation test: code workload only ---
# Run this AFTER the initialization cell to test a single
# workload in isolation with a fresh server.

config_baseline = SGLANG_TEST_CONFIGS[0]  # baseline

df_code_only = run_single_config(
    config_baseline,
    batch_sizes=[1, 8],
    workload_names=["code"],
)
print(df_code_only[["workload_name", "batch_size", "throughput_output_tok_s",
                     "ttft_p50_ms", "successful_requests", "failed_requests"]])

In [ ]:
print("=" * 60)
print("TEST 1: Baseline (RadixAttention ON)")
print("=" * 60)

config_baseline = SGLANG_TEST_CONFIGS[0]  # "baseline"
df_baseline = run_single_config(config_baseline)

df_baseline.to_csv("results_sglang_baseline.csv", index=False)
print(f"\nSaved {len(df_baseline)} rows -> results_sglang_baseline.csv")
df_baseline

## 7. Test 2 — No RadixAttention (Prefix Caching Disabled)

Starts a fresh SGLang server with `--disable-radix-cache`. This disables
SGLang's automatic prefix caching, providing a direct comparison to measure
RadixAttention's benefit — especially visible on the `shared_prefix` workload.

In [ ]:
print("=" * 60)
print("TEST 2: No RadixAttention (prefix caching disabled)")
print("=" * 60)

config_no_radix = SGLANG_TEST_CONFIGS[1]  # "no_radix"
df_no_radix = run_single_config(config_no_radix)

df_no_radix.to_csv("results_sglang_no_radix.csv", index=False)
print(f"\nSaved {len(df_no_radix)} rows -> results_sglang_no_radix.csv")
df_no_radix

## 8. Test 3 — Torch Compile

Starts a fresh SGLang server with `--enable-torch-compile`. This uses
PyTorch's `torch.compile` to JIT-compile the model, reducing Python
overhead and fusing kernels. Expect a **longer server startup** (compilation
happens on first few requests) but potentially higher steady-state throughput.

**Note**: If torch.compile is not supported for your SGLang version or model,
this test will report startup errors — the other tests remain unaffected.

In [ ]:
print("=" * 60)
print("TEST 3: Torch Compile")
print("=" * 60)

config_torch_compile = SGLANG_TEST_CONFIGS[2]  # "torch_compile"
df_torch_compile = run_single_config(config_torch_compile)

df_torch_compile.to_csv("results_sglang_torch_compile.csv", index=False)
print(f"\nSaved {len(df_torch_compile)} rows -> results_sglang_torch_compile.csv")
df_torch_compile

---
## 9. Combine Results & Compute Speedups

Merges the three per-config DataFrames, computes speedup-vs-baseline for
every row, and saves the combined CSV.

In [ ]:
import time, pandas as pd, numpy as np

# Combine
df_all = pd.concat([df_baseline, df_no_radix, df_torch_compile], ignore_index=True)

# Build baseline throughput lookup
bl = df_baseline.set_index(["workload_name", "batch_size"])["throughput_output_tok_s"]

def _speedup(row):
    key = (row["workload_name"], row["batch_size"])
    base = bl.get(key, None)
    if base and base > 0:
        return round(row["throughput_output_tok_s"] / base, 3)
    return np.nan

df_all["speedup_vs_baseline"] = df_all.apply(_speedup, axis=1)

ts = time.strftime("%Y%m%d_%H%M%S")
combined_csv = f"results_sglang_combined_{ts}.csv"
df_all.to_csv(combined_csv, index=False)
print(f"Combined {len(df_all)} rows -> {combined_csv}")
print(f"Configs : {sorted(df_all['config_name'].unique())}")
print(f"Workloads: {sorted(df_all['workload_name'].unique())}")
df_all[["config_name", "workload_name", "batch_size",
        "throughput_output_tok_s", "ttft_p50_ms", "itl_p50_ms",
        "speedup_vs_baseline"]]

---
# Comparison & Visualization

The cells below graph every key metric across the three configurations
and five workloads.  All plots pull from `df_all`.

## 10. Throughput Comparison

Output tokens/second for each workload, broken down by batch size and
configuration. Higher is better.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid", font_scale=1.05)
PALETTE = sns.color_palette("Set2", n_colors=df_all["config_name"].nunique())
CONFIG_ORDER = sorted(df_all["config_name"].unique())

workloads = sorted(df_all["workload_name"].unique())
n_wl = len(workloads)
ncols = min(3, n_wl)
nrows = (n_wl + ncols - 1) // ncols

fig, axes = plt.subplots(nrows, ncols, figsize=(6 * ncols, 5 * nrows),
                         squeeze=False)

for idx, wl in enumerate(workloads):
    ax = axes[idx // ncols][idx % ncols]
    sub = df_all[df_all["workload_name"] == wl].copy()
    pivot = sub.pivot_table(
        index="batch_size", columns="config_name",
        values="throughput_output_tok_s", aggfunc="first",
    ).reindex(columns=CONFIG_ORDER)
    pivot.plot.bar(ax=ax, color=PALETTE[:len(CONFIG_ORDER)], edgecolor="black",
                   linewidth=0.4)
    ax.set_title(wl, fontweight="bold")
    ax.set_ylabel("Output tok/s")
    ax.set_xlabel("Batch Size")
    ax.tick_params(axis="x", rotation=0)
    ax.legend(title="Config", fontsize=8, title_fontsize=9)

# hide unused subplots
for idx in range(n_wl, nrows * ncols):
    axes[idx // ncols][idx % ncols].set_visible(False)

fig.suptitle("SGLang Throughput Comparison  (output tok/s, higher is better)",
             fontsize=15, fontweight="bold", y=1.01)
plt.tight_layout()
plt.savefig("plot_sglang_throughput.png", dpi=150, bbox_inches="tight")
plt.show()

## 11. Time-to-First-Token (TTFT) Comparison

TTFT at p50, p95, and p99 for each workload.  Lower is better.

In [ ]:
fig, axes = plt.subplots(nrows, ncols, figsize=(6 * ncols, 5 * nrows),
                         squeeze=False)

for idx, wl in enumerate(workloads):
    ax = axes[idx // ncols][idx % ncols]
    sub = df_all[df_all["workload_name"] == wl].copy()

    # p50 bars
    pivot50 = sub.pivot_table(
        index="batch_size", columns="config_name",
        values="ttft_p50_ms", aggfunc="first",
    ).reindex(columns=CONFIG_ORDER)
    bars = pivot50.plot.bar(ax=ax, color=PALETTE[:len(CONFIG_ORDER)],
                            edgecolor="black", linewidth=0.4)

    # overlay p95 as error caps
    pivot95 = sub.pivot_table(
        index="batch_size", columns="config_name",
        values="ttft_p95_ms", aggfunc="first",
    ).reindex(columns=CONFIG_ORDER)
    for j, cfg in enumerate(CONFIG_ORDER):
        if cfg in pivot50.columns and cfg in pivot95.columns:
            x_positions = [p.get_x() + p.get_width() / 2
                           for p in bars.patches[j::len(CONFIG_ORDER)]]
            y_lo = pivot50[cfg].values
            y_hi = pivot95[cfg].values
            yerr = np.clip(y_hi - y_lo, 0, None)
            ax.errorbar(x_positions[:len(y_lo)], y_lo, yerr=yerr, fmt="none",
                        ecolor="gray", capsize=3, capthick=0.8, linewidth=0.8)

    ax.set_title(wl, fontweight="bold")
    ax.set_ylabel("TTFT (ms)")
    ax.set_xlabel("Batch Size")
    ax.tick_params(axis="x", rotation=0)
    ax.legend(title="Config", fontsize=8, title_fontsize=9)

for idx in range(n_wl, nrows * ncols):
    axes[idx // ncols][idx % ncols].set_visible(False)

fig.suptitle("SGLang Time to First Token  (p50 bars + p95 caps, lower is better)",
             fontsize=15, fontweight="bold", y=1.01)
plt.tight_layout()
plt.savefig("plot_sglang_ttft.png", dpi=150, bbox_inches="tight")
plt.show()

## 12. Inter-Token Latency (ITL) Comparison

ITL p50 with p95 error caps. Lower is better for streaming applications.

In [ ]:
fig, axes = plt.subplots(nrows, ncols, figsize=(6 * ncols, 5 * nrows),
                         squeeze=False)

for idx, wl in enumerate(workloads):
    ax = axes[idx // ncols][idx % ncols]
    sub = df_all[df_all["workload_name"] == wl].copy()

    pivot50 = sub.pivot_table(
        index="batch_size", columns="config_name",
        values="itl_p50_ms", aggfunc="first",
    ).reindex(columns=CONFIG_ORDER)
    bars = pivot50.plot.bar(ax=ax, color=PALETTE[:len(CONFIG_ORDER)],
                            edgecolor="black", linewidth=0.4)

    pivot95 = sub.pivot_table(
        index="batch_size", columns="config_name",
        values="itl_p95_ms", aggfunc="first",
    ).reindex(columns=CONFIG_ORDER)
    for j, cfg in enumerate(CONFIG_ORDER):
        if cfg in pivot50.columns and cfg in pivot95.columns:
            x_positions = [p.get_x() + p.get_width() / 2
                           for p in bars.patches[j::len(CONFIG_ORDER)]]
            y_lo = pivot50[cfg].values
            y_hi = pivot95[cfg].values
            yerr = np.clip(y_hi - y_lo, 0, None)
            ax.errorbar(x_positions[:len(y_lo)], y_lo, yerr=yerr, fmt="none",
                        ecolor="gray", capsize=3, capthick=0.8, linewidth=0.8)

    ax.set_title(wl, fontweight="bold")
    ax.set_ylabel("ITL (ms)")
    ax.set_xlabel("Batch Size")
    ax.tick_params(axis="x", rotation=0)
    ax.legend(title="Config", fontsize=8, title_fontsize=9)

for idx in range(n_wl, nrows * ncols):
    axes[idx // ncols][idx % ncols].set_visible(False)

fig.suptitle("SGLang Inter-Token Latency  (p50 bars + p95 caps, lower is better)",
             fontsize=15, fontweight="bold", y=1.01)
plt.tight_layout()
plt.savefig("plot_sglang_itl.png", dpi=150, bbox_inches="tight")
plt.show()

## 13. Speedup vs Baseline

Heatmaps showing how much faster (or slower) each config is relative to
the baseline (RadixAttention ON). Values > 1.0 mean the config is faster.

In [ ]:
spec_configs = [c for c in CONFIG_ORDER if c != "baseline"]
n_spec = len(spec_configs)

if n_spec > 0:
    fig, axes = plt.subplots(1, n_spec, figsize=(7 * n_spec, 5), squeeze=False)

    for i, cfg in enumerate(spec_configs):
        ax = axes[0][i]
        sub = df_all[df_all["config_name"] == cfg].copy()
        pivot = sub.pivot_table(
            index="batch_size", columns="workload_name",
            values="speedup_vs_baseline", aggfunc="first",
        )
        # Sort columns for consistency
        pivot = pivot.reindex(columns=sorted(pivot.columns))

        sns.heatmap(
            pivot, annot=True, fmt=".2f", cmap="RdYlGn", center=1.0,
            linewidths=0.5, ax=ax, vmin=0.5, vmax=2.0,
            cbar_kws={"label": "Speedup"},
        )
        ax.set_title(f"{cfg}", fontweight="bold")
        ax.set_ylabel("Batch Size")
        ax.set_xlabel("Workload")

    fig.suptitle("Speedup vs Baseline  (green > 1 = faster)",
                 fontsize=15, fontweight="bold", y=1.02)
    plt.tight_layout()
    plt.savefig("plot_sglang_speedup_heatmap.png", dpi=150, bbox_inches="tight")
    plt.show()
else:
    print("No non-baseline configs to compare.")

## 14. Latency vs Throughput Tradeoff

Scatter plot of TTFT (latency) against throughput. Ideal configs appear
in the **bottom-right** (high throughput, low latency).

In [ ]:
fig, ax = plt.subplots(figsize=(10, 7))

markers = ["o", "s", "D", "^", "v"]
wl_markers = {wl: markers[i % len(markers)] for i, wl in enumerate(workloads)}
cfg_colors = {c: PALETTE[i] for i, c in enumerate(CONFIG_ORDER)}

for _, row in df_all.iterrows():
    tp = row["throughput_output_tok_s"]
    ttft = row["ttft_p50_ms"]
    if pd.isna(tp) or pd.isna(ttft) or tp <= 0:
        continue
    ax.scatter(
        tp, ttft,
        c=[cfg_colors[row["config_name"]]],
        marker=wl_markers[row["workload_name"]],
        s=30 + row["batch_size"] * 1.5,
        alpha=0.75, edgecolors="black", linewidth=0.4,
    )

# Legends
from matplotlib.lines import Line2D
cfg_handles = [
    Line2D([0], [0], marker="o", color="w", markerfacecolor=cfg_colors[c],
           markersize=8, markeredgecolor="black", linewidth=0.4, label=c)
    for c in CONFIG_ORDER
]
wl_handles = [
    Line2D([0], [0], marker=wl_markers[w], color="w", markerfacecolor="gray",
           markersize=8, markeredgecolor="black", linewidth=0.4, label=w)
    for w in workloads
]
legend1 = ax.legend(handles=cfg_handles, title="Config", loc="upper left",
                     fontsize=8, title_fontsize=9)
ax.add_artist(legend1)
ax.legend(handles=wl_handles, title="Workload", loc="upper right",
          fontsize=8, title_fontsize=9)

ax.set_xlabel("Output Throughput (tok/s)", fontsize=12)
ax.set_ylabel("TTFT p50 (ms)  [lower is better]", fontsize=12)
ax.set_title("SGLang Latency vs Throughput Tradeoff  (bottom-right is ideal)",
             fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig("plot_sglang_latency_vs_throughput.png", dpi=150, bbox_inches="tight")
plt.show()

## 15. Goodput & Reliability

Compares raw throughput vs goodput (successful tokens only) and shows
success rates across configurations.

In [ ]:
goodput_col = "goodput_goodput_tok_s"
success_col = "goodput_success_rate"
sla_col = "goodput_sla_compliance_rate"

has_goodput = goodput_col in df_all.columns and df_all[goodput_col].notna().any()
has_success = success_col in df_all.columns and df_all[success_col].notna().any()

if has_goodput or has_success:
    n_panels = sum([has_goodput, has_success, sla_col in df_all.columns])
    fig, axes = plt.subplots(1, n_panels, figsize=(6 * n_panels, 5), squeeze=False)
    panel = 0

    # -- Raw throughput vs Goodput --
    if has_goodput:
        ax = axes[0][panel]
        panel += 1
        compare = df_all.groupby("config_name").agg(
            raw_tp=("throughput_output_tok_s", "mean"),
            goodput=(goodput_col, "mean"),
        )
        compare.plot.bar(ax=ax, color=[PALETTE[0], PALETTE[1]],
                         edgecolor="black", linewidth=0.4)
        ax.set_title("Avg Raw Throughput vs Goodput", fontweight="bold")
        ax.set_ylabel("tok/s")
        ax.tick_params(axis="x", rotation=15)

    # -- Success rate by config + workload --
    if has_success:
        ax = axes[0][panel]
        panel += 1
        pivot_sr = df_all.pivot_table(
            index="workload_name", columns="config_name",
            values=success_col, aggfunc="mean",
        ).reindex(columns=CONFIG_ORDER)
        pivot_sr.plot.bar(ax=ax, color=PALETTE[:len(CONFIG_ORDER)],
                          edgecolor="black", linewidth=0.4)
        ax.set_title("Success Rate by Workload", fontweight="bold")
        ax.set_ylabel("Success Rate (%)")
        ax.set_ylim(0, 105)
        ax.axhline(100, color="green", linestyle="--", alpha=0.4)
        ax.tick_params(axis="x", rotation=25)

    # -- SLA compliance --
    if sla_col in df_all.columns and df_all[sla_col].notna().any():
        ax = axes[0][panel]
        panel += 1
        pivot_sla = df_all.pivot_table(
            index="workload_name", columns="config_name",
            values=sla_col, aggfunc="mean",
        ).reindex(columns=CONFIG_ORDER)
        pivot_sla.plot.bar(ax=ax, color=PALETTE[:len(CONFIG_ORDER)],
                           edgecolor="black", linewidth=0.4)
        ax.set_title("SLA Compliance (TTFT < 500 ms)", fontweight="bold")
        ax.set_ylabel("Compliance Rate (%)")
        ax.set_ylim(0, 105)
        ax.axhline(100, color="green", linestyle="--", alpha=0.4)
        ax.tick_params(axis="x", rotation=25)

    fig.suptitle("SGLang Goodput & Reliability", fontsize=15, fontweight="bold", y=1.02)
    plt.tight_layout()
    plt.savefig("plot_sglang_goodput.png", dpi=150, bbox_inches="tight")
    plt.show()
else:
    print("No goodput/success-rate data available.")

## 16. KV Cache Analysis

KV cache utilization and prefix-cache hit rates (when available).
RadixAttention's benefit should be visible as higher prefix cache hit rates
in the baseline config vs no_radix.

In [ ]:
kv_util_col = "kv_cache_utilization_pct"
prefix_col = "kv_prefix_cache_hit_rate"

has_kv = kv_util_col in df_all.columns and df_all[kv_util_col].notna().any()
has_prefix = prefix_col in df_all.columns and df_all[prefix_col].notna().any()

if has_kv or has_prefix:
    n_panels = sum([has_kv, has_prefix])
    fig, axes = plt.subplots(1, n_panels, figsize=(7 * n_panels, 5), squeeze=False)
    panel = 0

    if has_kv:
        ax = axes[0][panel]
        panel += 1
        pivot_kv = df_all.pivot_table(
            index="batch_size", columns="config_name",
            values=kv_util_col, aggfunc="mean",
        ).reindex(columns=CONFIG_ORDER)
        pivot_kv.plot.bar(ax=ax, color=PALETTE[:len(CONFIG_ORDER)],
                          edgecolor="black", linewidth=0.4)
        ax.set_title("KV Cache Utilization by Batch Size", fontweight="bold")
        ax.set_ylabel("Utilization (%)")
        ax.set_xlabel("Batch Size")
        ax.tick_params(axis="x", rotation=0)

    if has_prefix:
        ax = axes[0][panel]
        panel += 1
        pivot_prefix = df_all.pivot_table(
            index="workload_name", columns="config_name",
            values=prefix_col, aggfunc="mean",
        ).reindex(columns=CONFIG_ORDER)
        pivot_prefix.plot.bar(ax=ax, color=PALETTE[:len(CONFIG_ORDER)],
                              edgecolor="black", linewidth=0.4)
        ax.set_title("Prefix Cache Hit Rate by Workload", fontweight="bold")
        ax.set_ylabel("Hit Rate (%)")
        ax.set_xlabel("Workload")
        ax.tick_params(axis="x", rotation=25)

    fig.suptitle("SGLang KV Cache Analysis", fontsize=15, fontweight="bold", y=1.02)
    plt.tight_layout()
    plt.savefig("plot_sglang_kv_cache.png", dpi=150, bbox_inches="tight")
    plt.show()
else:
    print("No KV cache metrics available.")

## 17. Request Latency Distribution

Mean request latency across workloads, grouped by config and batch size.

In [ ]:
lat_col = "request_latency_mean_ms"

if lat_col in df_all.columns and df_all[lat_col].notna().any():
    fig, axes = plt.subplots(nrows, ncols, figsize=(6 * ncols, 5 * nrows),
                             squeeze=False)
    for idx, wl in enumerate(workloads):
        ax = axes[idx // ncols][idx % ncols]
        sub = df_all[df_all["workload_name"] == wl]
        pivot = sub.pivot_table(
            index="batch_size", columns="config_name",
            values=lat_col, aggfunc="first",
        ).reindex(columns=CONFIG_ORDER)
        pivot.plot.bar(ax=ax, color=PALETTE[:len(CONFIG_ORDER)],
                       edgecolor="black", linewidth=0.4)
        ax.set_title(wl, fontweight="bold")
        ax.set_ylabel("Mean Request Latency (ms)")
        ax.set_xlabel("Batch Size")
        ax.tick_params(axis="x", rotation=0)
        ax.legend(title="Config", fontsize=8, title_fontsize=9)

    for idx in range(n_wl, nrows * ncols):
        axes[idx // ncols][idx % ncols].set_visible(False)

    fig.suptitle("SGLang Mean Request Latency  (lower is better)",
                 fontsize=15, fontweight="bold", y=1.01)
    plt.tight_layout()
    plt.savefig("plot_sglang_request_latency.png", dpi=150, bbox_inches="tight")
    plt.show()
else:
    print("No request latency data available.")

## 18. Scaling Efficiency

Line plots showing how throughput scales with batch size for each config.
Ideal scaling would be linear.

In [ ]:
fig, axes = plt.subplots(nrows, ncols, figsize=(6 * ncols, 5 * nrows),
                         squeeze=False)

for idx, wl in enumerate(workloads):
    ax = axes[idx // ncols][idx % ncols]
    sub = df_all[df_all["workload_name"] == wl]

    for i, cfg in enumerate(CONFIG_ORDER):
        cfg_sub = sub[sub["config_name"] == cfg].sort_values("batch_size")
        if cfg_sub.empty:
            continue
        ax.plot(
            cfg_sub["batch_size"], cfg_sub["throughput_output_tok_s"],
            marker="o", label=cfg, color=PALETTE[i], linewidth=2,
            markersize=6,
        )

    ax.set_title(wl, fontweight="bold")
    ax.set_ylabel("Output tok/s")
    ax.set_xlabel("Batch Size")
    ax.set_xscale("log", base=2)
    ax.legend(title="Config", fontsize=8, title_fontsize=9)
    ax.grid(True, alpha=0.3)

for idx in range(n_wl, nrows * ncols):
    axes[idx // ncols][idx % ncols].set_visible(False)

fig.suptitle("SGLang Throughput Scaling with Batch Size",
             fontsize=15, fontweight="bold", y=1.01)
plt.tight_layout()
plt.savefig("plot_sglang_scaling.png", dpi=150, bbox_inches="tight")
plt.show()

## 19. Comprehensive Summary

Best configuration per workload (by throughput and by TTFT), plus an
overall metrics comparison table.

In [ ]:
print("=" * 70)
print("BEST CONFIG PER WORKLOAD  (by output throughput)")
print("=" * 70)

for wl in workloads:
    sub = df_all[df_all["workload_name"] == wl]
    if sub.empty:
        continue
    best = sub.loc[sub["throughput_output_tok_s"].idxmax()]
    print(f"  {wl:>20s}:  {best['config_name']} "
          f"(batch={int(best['batch_size'])})  "
          f"{best['throughput_output_tok_s']:.1f} tok/s")

print()
print("=" * 70)
print("BEST CONFIG PER WORKLOAD  (by lowest TTFT p50)")
print("=" * 70)

for wl in workloads:
    sub = df_all[df_all["workload_name"] == wl].dropna(subset=["ttft_p50_ms"])
    if sub.empty:
        continue
    best = sub.loc[sub["ttft_p50_ms"].idxmin()]
    print(f"  {wl:>20s}:  {best['config_name']} "
          f"(batch={int(best['batch_size'])})  "
          f"{best['ttft_p50_ms']:.1f} ms")

print()
print("=" * 70)
print("OVERALL METRICS BY CONFIG  (averaged across all workloads & batches)")
print("=" * 70)

summary_metrics = ["throughput_output_tok_s", "ttft_p50_ms", "itl_p50_ms",
                   "request_latency_mean_ms", "speedup_vs_baseline"]
available = [m for m in summary_metrics if m in df_all.columns]

summary = df_all.groupby("config_name")[available].mean().round(2)
print(summary.to_string())

# Per-workload throughput pivot
print()
print("=" * 70)
print("THROUGHPUT PIVOT  (output tok/s, rows=workload+batch, cols=config)")
print("=" * 70)
tp_pivot = df_all.pivot_table(
    index=["workload_name", "batch_size"], columns="config_name",
    values="throughput_output_tok_s", aggfunc="first",
).reindex(columns=CONFIG_ORDER)
print(tp_pivot.to_string())

---
## 20. Export All Results

Downloads the per-config CSVs, combined CSV, and all plot PNGs.

In [ ]:
import glob
from google.colab import files

export_files = [
    "results_sglang_baseline.csv",
    "results_sglang_no_radix.csv",
    "results_sglang_torch_compile.csv",
]
# Pick up the combined CSV (timestamped)
export_files += sorted(glob.glob("results_sglang_combined_*.csv"))[-1:]
# Pick up all SGLang plots
export_files += sorted(glob.glob("plot_sglang_*.png"))

print("Downloading:")
for f in export_files:
    if os.path.exists(f):
        print(f"  {f}")
        files.download(f)
    else:
        print(f"  {f}  (not found, skipping)")